# Example 5: Comprehensive Quantization Comparison

This notebook provides a comprehensive side-by-side comparison of different quantization methods:
- Memory usage comparison
- Inference speed benchmarking
- Accuracy evaluation
- Visual comparison (if matplotlib available)

**Goal:** Help you choose the right quantization method for your use case!

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Try to import matplotlib
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
    print("✓ matplotlib available - will create visualizations")
except ImportError:
    HAS_MATPLOTLIB = False
    print("○ matplotlib not available - skipping plots")

## Create Benchmark Model

We'll use a simplified transformer-like model with ~100M parameters.

In [ ]:
class BenchmarkModel(nn.Module):
    """Simplified transformer-like model."""
    def __init__(self, d_model=1024, num_layers=12, vocab_size=50000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model * 4),
                nn.GELU(),
                nn.Linear(d_model * 4, d_model),
            )
            for _ in range(num_layers)
        ])
        self.output = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = x + layer(x)
        return self.output(x)

model = BenchmarkModel().to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {num_params:,}")
print(f"Approximately: {num_params/1e6:.0f}M parameters")

## Part 1: Memory Comparison

In [ ]:
def get_model_size(model):
    """Calculate model size in MB."""
    total_size = 0
    for param in model.parameters():
        total_size += param.nelement() * param.element_size()
    return total_size / 1e6

# Calculate sizes for different precisions
fp32_size = get_model_size(model)
fp16_size = fp32_size / 2
int8_size = fp32_size / 4
int4_size = fp32_size / 8

print("Memory Requirements:")
print(f"  FP32: {fp32_size:.2f} MB")
print(f"  FP16: {fp16_size:.2f} MB  (2× reduction)")
print(f"  INT8: {int8_size:.2f} MB  (4× reduction)")
print(f"  INT4: {int4_size:.2f} MB  (8× reduction)")

## Part 2: Real-World Impact

Let's see what this means for popular model sizes.

In [ ]:
import pandas as pd

model_sizes = {
    'Model': ['1B params', '7B params', '13B params', '70B params'],
    'FP32': ['4 GB', '28 GB', '52 GB', '280 GB'],
    'FP16': ['2 GB', '14 GB', '26 GB', '140 GB'],
    'INT8': ['1 GB', '7 GB', '13 GB', '70 GB'],
    'INT4': ['0.5 GB', '3.5 GB', '6.5 GB', '35 GB'],
}

df = pd.DataFrame(model_sizes)
print("\nMemory Requirements by Model Size:")
print(df.to_string(index=False))

print("\n💡 Practical Impact:")
print("  • FP32 LLaMA-7B: Requires A100 40GB")
print("  • FP16 LLaMA-7B: Fits on RTX 3090 (24GB)")
print("  • INT8 LLaMA-7B: Fits on RTX 3060 (12GB)")
print("  • INT4 LLaMA-7B: Fits on most GPUs (8GB+)")

## Part 3: Accuracy Comparison

Test how different quantization methods affect accuracy.

In [ ]:
# Create test data
batch_size = 32
seq_length = 128
test_input = torch.randint(0, 50000, (batch_size, seq_length), device=device)

# Get baseline output
model.eval()
with torch.no_grad():
    baseline_output = model(test_input)

print(f"Test input shape: {test_input.shape}")
print(f"Baseline output shape: {baseline_output.shape}")

In [ ]:
# Test FP16
model_fp16 = model.half()
with torch.no_grad():
    fp16_output = model_fp16(test_input)

# Calculate metrics
def calculate_accuracy(original, quantized):
    mse = torch.mean((original - quantized.float()) ** 2).item()
    
    # Top-1 agreement
    orig_pred = original.argmax(dim=-1)
    quant_pred = quantized.argmax(dim=-1)
    top1_agreement = (orig_pred == quant_pred).float().mean().item()
    
    # Cosine similarity
    cosine_sim = torch.nn.functional.cosine_similarity(
        original.flatten(),
        quantized.float().flatten(),
        dim=0
    ).item()
    
    return {
        'mse': mse,
        'top1_agreement': top1_agreement,
        'cosine_similarity': cosine_sim
    }

fp16_metrics = calculate_accuracy(baseline_output, fp16_output)
print("\nFP16 Accuracy:")
print(f"  Top-1 agreement: {fp16_metrics['top1_agreement']*100:.2f}%")
print(f"  Cosine similarity: {fp16_metrics['cosine_similarity']:.4f}")
print(f"  MSE: {fp16_metrics['mse']:.6f}")

## Part 4: Speed Comparison

Benchmark inference speed for different precisions.

In [ ]:
def benchmark_inference(model, input_data, num_iterations=50):
    """Benchmark model inference."""
    model.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(input_data)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    times = []
    with torch.no_grad():
        for _ in range(num_iterations):
            start = time.time()
            _ = model(input_data)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append(time.time() - start)
    
    return {
        'mean_time': np.mean(times),
        'std_time': np.std(times),
        'throughput': input_data.shape[0] / np.mean(times)
    }

# Benchmark FP32
model_fp32 = model.float()
fp32_bench = benchmark_inference(model_fp32, test_input)
print(f"FP32: {fp32_bench['mean_time']*1000:.2f} ms/batch, {fp32_bench['throughput']:.1f} samples/sec")

# Benchmark FP16
fp16_bench = benchmark_inference(model_fp16, test_input)
print(f"FP16: {fp16_bench['mean_time']*1000:.2f} ms/batch, {fp16_bench['throughput']:.1f} samples/sec")
print(f"      Speedup: {fp32_bench['mean_time'] / fp16_bench['mean_time']:.2f}x")

## Part 5: Summary Table

In [ ]:
summary = {
    'Method': ['FP32', 'FP16', 'INT8', 'INT4'],
    'Memory (MB)': [fp32_size, fp16_size, int8_size, int4_size],
    'Memory Reduction': ['1×', '2×', '4×', '8×'],
    'Typical Accuracy': ['100%', '99.9%+', '98-99%', '95-98%'],
    'Speed vs FP32': ['1.0×', '1.5-2×', '2-3×', '2-4×'],
    'Best Use Case': [
        'Training',
        'Inference (balanced)',
        'Memory-limited inference',
        'Maximum compression'
    ]
}

df_summary = pd.DataFrame(summary)
print("\n" + "="*100)
print("QUANTIZATION SUMMARY")
print("="*100)
print(df_summary.to_string(index=False))

## Part 6: Visualization

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    methods = ['FP32', 'FP16', 'INT8', 'INT4']
    memory = [fp32_size, fp16_size, int8_size, int4_size]
    speed = [1.0, 1.75, 2.5, 3.0]  # Typical speedups
    accuracy = [100, 99.9, 98.5, 96.5]  # Typical accuracy
    
    # Memory
    axes[0].bar(methods, memory, color=['blue', 'green', 'orange', 'red'])
    axes[0].set_ylabel('Memory (MB)')
    axes[0].set_title('Model Size Comparison')
    axes[0].tick_params(axis='x', rotation=0)
    
    # Speed
    axes[1].bar(methods, speed, color=['blue', 'green', 'orange', 'red'])
    axes[1].set_ylabel('Speedup (vs FP32)')
    axes[1].set_title('Inference Speed')
    axes[1].tick_params(axis='x', rotation=0)
    
    # Accuracy
    axes[2].bar(methods, accuracy, color=['blue', 'green', 'orange', 'red'])
    axes[2].set_ylabel('Accuracy (%)')
    axes[2].set_title('Typical Accuracy')
    axes[2].set_ylim([90, 100])
    axes[2].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig('quantization_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✓ Saved plot to: quantization_comparison.png")
else:
    print("Install matplotlib to see visualizations: pip install matplotlib")

## Recommendations

### Choose Based on Your Constraints:

**1. Maximum Accuracy Needed:**
- Use **FP16** (99.9%+ agreement, 2× memory savings)

**2. Memory Constrained (8-12GB GPU):**
- Use **INT8** with bitsandbytes (98%+ agreement, 4× savings)

**3. Severe Memory Constraints (4-8GB GPU):**
- Use **INT4/NF4** with bitsandbytes (95%+ agreement, 8× savings)

**4. Production Inference (Speed Critical):**
- Use **GPTQ** or **AWQ** (97%+ agreement, 2-3× faster)

**5. Fine-tuning with Limited Memory:**
- Use **QLoRA** with NF4 (enables fine-tuning 65B models on 24GB GPU)

### Real-World Examples:

```python
# Maximum quality
model = AutoModelForCausalLM.from_pretrained(
    "model-name",
    torch_dtype=torch.float16
)

# Memory-efficient inference
model = AutoModelForCausalLM.from_pretrained(
    "model-name",
    load_in_8bit=True
)

# Maximum compression
model = AutoModelForCausalLM.from_pretrained(
    "model-name",
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4"
)
```

## 🎉 Tutorial Complete!

### What You've Learned:
1. ✅ Fundamentals of quantization (Notebook 1)
2. ✅ Advanced 4-bit techniques (Notebook 2)
3. ✅ Real LLM quantization (Notebook 3)
4. ✅ GPTQ for production (Notebook 4)
5. ✅ Comprehensive comparison (Notebook 5)

### Next Steps:
- Experiment with your own models
- Try different quantization configs
- Explore QLoRA for fine-tuning
- Deploy quantized models in production

### Resources:
- HuggingFace: https://huggingface.co/docs/transformers
- bitsandbytes: https://github.com/TimDettmers/bitsandbytes
- Auto-GPTQ: https://github.com/PanQiWei/AutoGPTQ
- QLoRA paper: https://arxiv.org/abs/2305.14314